# CascadeFlow Test-Set Evaluation

Run the Which VLM router test set through CascadeFlow using the locally hosted VLMs to compare routing accuracy, latency, and spend.

**Workflow**

1. Configure dataset paths, cascade parameters, and local VLM endpoints.
2. Run CascadeFlow over the sampled test records.
3. Score each response against the ground truth to compute accuracy metrics.
4. Persist the telemetry for downstream analysis.

In [1]:
from pathlib import Path

import logging
import pandas as pd

from cascadeflow_experiment import (
    ExperimentConfig,
    ModelSpec,
    create_default_config,
    persist_results,
    run_experiment_with_logging,
    summarize_results,
)
from which_vlm.dataset_builder.evaluation import Scorer

### Configure dataset + cascade

Update the dataset path, sampling options, and list of local VLM deployments that CascadeFlow should consider. Costs are specified in USD per 1K tokens.

In [2]:
config = create_default_config()
config.dataset_path = config.project_root / "dataset/final_dataset_1127/router_sample_records_test.parquet"
config.max_samples = 200  # reduce if you only want a smoke test
config.seed = 13
config.generation_max_tokens = 300
config.generation_temperature = 0.1
config.experiment_name = "cascadeflow_router_test"
config.verbose_agent = False

# Update the list below so every local deployment gets its own ModelSpec.
# Example showing five staged endpoints—swap base_url/cost/temperature to match your setup.
CASCADE_MODELS = [
    ModelSpec(
        name="PatronusAI/glider",
        base_url="http://localhost:8805/v1",
        cost=0.00045,
        provider="vllm",
        temperature=0.1,
        quality_score=0.65,
        speed_ms=600,
        keywords=["vlm", "fast"],
        domains=["diagram_reasoning", "general"],
    ),
    ModelSpec(
        name="PatronusAI/glider-pro",
        base_url="http://localhost:8806/v1",
        cost=0.0009,
        provider="vllm",
        temperature=0.15,
        quality_score=0.8,
        speed_ms=950,
        keywords=["vlm", "balanced"],
        domains=["diagram_reasoning", "math"],
    ),
    ModelSpec(
        name="PatronusAI/glider-max",
        base_url="http://localhost:8807/v1",
        cost=0.0018,
        provider="vllm",
        temperature=0.2,
        quality_score=0.9,
        speed_ms=1400,
        keywords=["vlm", "high_accuracy"],
        domains=["diagram_reasoning", "ocr", "math"],
    ),
    ModelSpec(
        name="PatronusAI/glider-ultra",
        base_url="http://localhost:8808/v1",
        cost=0.0024,
        provider="vllm",
        temperature=0.2,
        quality_score=0.92,
        speed_ms=1600,
        keywords=["vlm", "deep_reasoning"],
        domains=["diagram_reasoning", "ocr", "math", "science"],
    ),
    ModelSpec(
        name="PatronusAI/glider-research",
        base_url="http://localhost:8809/v1",
        cost=0.003,
        provider="vllm",
        temperature=0.25,
        quality_score=0.95,
        speed_ms=1900,
        keywords=["vlm", "research"],
        domains=["diagram_reasoning", "ocr", "math", "science"],
    ),
]

config.cascade_models = CASCADE_MODELS
config

ExperimentConfig(dataset_path=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset_1127/router_sample_records_test.parquet'), cauldron_lookup_path=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/processed/cauldron_poc_multi.parquet'), image_root=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/images/cauldron'), output_dir=PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/results'), cascade_models=[ModelSpec(name='PatronusAI/glider', base_url='http://localhost:8805/v1', cost=0.00045, provider='vllm', temperature=0.1, max_tokens=None, system_prompt=None, quality_score=0.65, speed_ms=600, keywords=['vlm', 'fast'], domains=['diagram_reasoning', 'general'], extra={}), ModelSpe

### Optional: inspect available VLM deployments

Use a quick helper to make sure the vLLM/OpenAI-compatible server is running and exposes the expected models.

In [3]:
import requests
from pprint import pprint


def list_vllm_models(base_url: str = "http://localhost:8805/v1") -> list[str]:
    response = requests.get(f"{base_url}/models", timeout=5)
    response.raise_for_status()
    payload = response.json()
    return [item["id"] for item in payload.get("data", [])]


try:
    model_ids = list_vllm_models()
    print("Models exposed by http://localhost:8805/v1:")
    pprint(model_ids)
except Exception as exc:
    print(f"Skipping model probe: {exc}")

Models exposed by http://localhost:8805/v1:
['PatronusAI/glider']


### Load ground-truth metadata

We only need the identifiers and labels so the read is fast even for the full test set.

In [4]:
GT_COLUMNS = ["sample_id", "ground_truth", "ground_truth_type"]

ground_truth_df = pd.read_parquet(config.dataset_path, columns=GT_COLUMNS)
print(f"Loaded {len(ground_truth_df)} labeled samples from {config.dataset_path.name}")
ground_truth_df.head(2)

Loaded 21795 labeled samples from router_sample_records_test.parquet


,sample_id,ground_truth,ground_truth_type
0,ai2d_00026_baeba96290d54334,Answer: A,mc
1,ai2d_00028_e1bae1fe1bb31d0d,Answer: A,mc


### Run CascadeFlow over the sampled rows

This step issues one or more requests per sample, so expect it to take several minutes depending on how many models are in the cascade.

In [5]:
%%time
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
)

results_df, samples_df = run_experiment_with_logging(
    config,
    logger=logging.getLogger('cascadeflow.notebook'),
    suppress_hf_logs=True,
    num_workers=4,
    chunk_size=40,
)
results_df.head()

2025-11-29 17:55:31,570 | INFO | Preparing samples for CascadeFlow...


['sample_id', 'run_id', 'timestamp_utc', 'image_path', 'image_bytes_hash', 'prompt_raw', 'prompt_formatted', 'system_prompt', 'source_dataset', 'source_config', 'router_task', 'ground_truth', 'ground_truth_type', 'mc_options', 'source_index', 'img_width', 'img_height', 'img_aspect_ratio', 'img_file_size_bytes', 'txt_prompt_length_chars', 'txt_prompt_length_words', 'txt_question_type', 'txt_has_mc_options', 'model_name', 'model_id', 'response_raw', 'response_parsed', 'response_length_chars', 'response_length_tokens', 'stop_reason', 'error_message', 'is_refusal', 'ok', 'score_exact_match', 'score_exact_match_normalized', 'score_contains_gt', 'score_gt_in_response', 'score_f1', 'score_numeric_match', 'score_mc_letter_match', 'is_correct', 'pred_answer_letter', 'gt_answer_letter', 'input_tokens', 'output_tokens', 'total_tokens', 'latency_ms', 'estimated_cost_usd', 'inference_temperature', 'inference_max_tokens', 'inference_top_p', 'semantic_f1_precision', 'semantic_f1_recall', 'semantic_f1

2025-11-29 17:55:31,980 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2025-11-29 17:55:32,003 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceM4/the_cauldron/847a98a779b1652d65111daf20c972dfcd333605/README.md "HTTP/1.1 200 OK"
2025-11-29 17:55:32,054 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/847a98a779b1652d65111daf20c972dfcd333605/the_cauldron.py "HTTP/1.1 404 Not Found"
2025-11-29 17:55:32,249 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceM4/the_cauldron/HuggingFaceM4/the_cauldron.py "HTTP/1.1 404 Not Found"
2025-11-29 17:55:32,339 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceM4/the_cauldron/revision/847a98a779b1652d65111daf20c972dfcd333605 "HTTP/1.1 200 OK"
2025-11-29 17:55:32,417 | INFO | HTTP Request: HEA

CPU times: user 35.9 s, sys: 8.61 s, total: 44.5 s
Wall time: 1min 19s


KeyboardInterrupt: 

### Built-in CascadeFlow summary tables

In [ ]:
summary_tables = summarize_results(results_df)
summary_tables

### Compute accuracy + cost metrics

We reuse the Which-VLM scoring helpers so the metrics line up with the offline evaluation pipeline.

In [ ]:
def score_prediction(row):
    return Scorer.compute_all_scores(
        pred=row.get("raw_response") or "",
        gt=row.get("ground_truth") or "",
        gt_type=row.get("ground_truth_type") or "exact",
    )


scored_df = results_df.merge(ground_truth_df, on="sample_id", how="left")
score_columns = scored_df.apply(score_prediction, axis=1).apply(pd.Series)
scored_df = pd.concat([scored_df, score_columns], axis=1)

successful_df = scored_df[scored_df["error"].isna()].copy()
if successful_df.empty:
    raise RuntimeError("No successful samples to score; inspect `results_df` for transport errors.")

router_metrics = pd.DataFrame([
    {
        "num_samples": len(scored_df),
        "successful_routes": len(successful_df),
        "router_accuracy": successful_df["is_correct"].mean(),
        "avg_cost_per_query": successful_df["total_cost"].mean(),
        "total_cost": successful_df["total_cost"].sum(),
        "avg_latency_ms": successful_df["latency_ms"].mean(),
    }
])
router_metrics

### Model-level breakdown

In [ ]:
per_model_breakdown = (
    successful_df.groupby("model_used")[
        ["sample_id", "is_correct", "total_cost", "latency_ms"]
    ]
    .agg(
        samples=("sample_id", "count"),
        accuracy=("is_correct", "mean"),
        avg_cost=("total_cost", "mean"),
        total_cost=("total_cost", "sum"),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .sort_values("samples", ascending=False)
)
per_model_breakdown

### Persist raw + scored telemetry

In [ ]:
artifact_paths = persist_results(scored_df, config)
artifact_paths